# Step 5 - Hyperparameter Optimisation (Optuna)

Optuna TPE search for the four ensemble models (Random Forest, XGBoost, LightGBM, CatBoost). Each objective maximises mean F1 under stratified 5-fold CV on the training set only, then tuned models are evaluated on the held-out test set and compared to defaults.

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

def _find_root():
    p = Path.cwd().resolve()
    for cand in [p, *p.parents]:
        if (cand / "src").is_dir() and (cand / "data").is_dir():
            return cand
    return p

ROOT = _find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from src import config as C
from src import viz
viz.setup_style()
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
print("Repository root:", ROOT)


Repository root: /Users/rauankaztaev/IdeaProjects/Draft/project


In [2]:
from src import data, hpo, models, benchmark
from src import utils

df = data.load_clean()
x_train, x_test, y_train, y_test = data.get_splits(df)
N_TRIALS = 30  # increase for a finer search; kept modest for reproducible runtime
best = hpo.optimise(x_train, y_train, n_trials=N_TRIALS)
hpo.save_best_params(best)
for name, info in best.items():
    print(f"{name}: CV-F1={info['best_value']:.4f}")
    print("   params:", info["best_params"])

Random Forest: CV-F1=0.9984
   params: {'n_estimators': 350, 'max_depth': 16, 'min_samples_split': 16, 'min_samples_leaf': 2, 'max_features': 'sqrt'}
XGBoost: CV-F1=0.9979
   params: {'n_estimators': 300, 'max_depth': 10, 'learning_rate': 0.1205712628744377, 'subsample': 0.8394633936788146, 'colsample_bytree': 0.6624074561769746, 'min_child_weight': 2, 'gamma': 0.2904180608409973}
LightGBM: CV-F1=0.9986
   params: {'n_estimators': 600, 'num_leaves': 99, 'max_depth': 15, 'learning_rate': 0.06475722446081826, 'subsample': 0.7989679156491106, 'colsample_bytree': 0.7170154772685936, 'min_child_samples': 20}
CatBoost: CV-F1=0.9988
   params: {'iterations': 600, 'depth': 6, 'learning_rate': 0.03375175709406991, 'l2_leaf_reg': 1.6312372238875648}


## 4.1 Evaluate tuned models on the held-out test set

In [3]:
tuned_est = hpo.build_tuned_estimators(best)
tuned_pipes = {name: models.build_pipeline(name, est, x_train)
               for name, est in tuned_est.items()}
tuned_results, tuned_fitted, _ = benchmark.evaluate_on_test(
    x_train, y_train, x_test, y_test, tuned_pipes)
display(tuned_results.round(4))
for name, est in tuned_fitted.items():
    utils.save_model(est, f"tuned_{name}")
utils.save_table(tuned_results.round(4), "benchmark_tuned",
                 caption="Held-out performance after Optuna tuning.", label="tab:tuned")

,Model,Accuracy,Balanced Accuracy,Precision,Recall,F1,ROC AUC,PR AUC,MCC,Cohen Kappa,Log Loss,Brier Score,Train Time (s),Predict Time (s)
0,CatBoost,0.9994,0.9995,0.9986,1.0000,0.9993,0.9999,0.9999,0.9987,0.9987,0.0052,0.0009,0.6771,0.0031
1,XGBoost,0.9982,0.9980,0.9986,0.9972,0.9979,1.0000,0.9999,0.9962,0.9962,0.0077,0.0016,0.2240,0.0036
2,LightGBM,0.9982,0.9980,0.9986,0.9972,0.9979,1.0000,1.0000,0.9962,0.9962,0.0123,0.0019,1.6714,0.0112
3,Random Forest,0.9969,0.9966,0.9986,0.9944,0.9965,1.0000,1.0000,0.9937,0.9937,0.0110,0.0021,0.2503,0.0281


{'csv': PosixPath('/Users/rauankaztaev/IdeaProjects/Draft/project/tables/benchmark_tuned.csv'),
 'tex': PosixPath('/Users/rauankaztaev/IdeaProjects/Draft/project/tables/benchmark_tuned.tex')}

## 4.2 Default vs tuned comparison

In [4]:
default_results = pd.read_csv(C.TABLES_DIR / "benchmark.csv")
rows = []
for name in tuned_results["Model"]:
    d = default_results.loc[default_results["Model"] == name].iloc[0]
    t = tuned_results.loc[tuned_results["Model"] == name].iloc[0]
    rows.append({"Model": name,
                 "F1_default": round(d["F1"], 4), "F1_tuned": round(t["F1"], 4),
                 "dF1": round(t["F1"] - d["F1"], 4),
                 "ROC_AUC_default": round(d["ROC AUC"], 4), "ROC_AUC_tuned": round(t["ROC AUC"], 4)})
comp = pd.DataFrame(rows)
display(comp)
utils.save_table(comp, "tuning_comparison",
                 caption="Default vs Optuna-tuned ensemble performance.", label="tab:tuningcomp")

,Model,F1_default,F1_tuned,dF1,ROC_AUC_default,ROC_AUC_tuned
0,CatBoost,0.9979,0.9993,0.0014,0.9999,0.9999
1,XGBoost,0.9979,0.9979,-0.0000,0.9999,1.0000
2,LightGBM,0.9965,0.9979,0.0014,0.9999,1.0000
3,Random Forest,0.9965,0.9965,-0.0000,1.0000,1.0000


{'csv': PosixPath('/Users/rauankaztaev/IdeaProjects/Draft/project/tables/tuning_comparison.csv'),
 'tex': PosixPath('/Users/rauankaztaev/IdeaProjects/Draft/project/tables/tuning_comparison.tex')}

**Observation.** Because the default ensembles already operate near the performance ceiling on this dataset, tuning yields only marginal changes in F1/AUC. This is itself informative: the task is easy enough that careful hyperparameter search is not the bottleneck; data realism is. The tuned configurations are saved for reproducibility.